In [ ]:
# Manual Kaggle acceptance only. Run cells deliberately; no result is prefilled.
import os, sys, json, hashlib, pathlib, subprocess, textwrap, zipfile, time, signal
from pathlib import Path
PYTHON = "/usr/bin/python3.12"
assert Path(PYTHON).is_file(), "Kaggle CPython 3.12 is missing"
print({"notebook_kernel":sys.executable,"runtime_python":subprocess.check_output([PYTHON,"-c","import sys;print(sys.executable,sys.version)"]).decode().strip()})


In [ ]:
# Inspect exact hardware and mounted datasets before any installation.
sm = subprocess.run(["nvidia-smi","--query-gpu=name,compute_cap","--format=csv,noheader"],capture_output=True,text=True,check=True,timeout=15)
GPUS = [tuple(x.strip() for x in row.split(",",1)) for row in sm.stdout.splitlines() if row.strip()]
assert GPUS == [("NVIDIA L4","8.9")]*4, GPUS
INPUT = Path("/kaggle/input")
print("GPUS",GPUS)
print("INPUTS",[str(p) for p in INPUT.iterdir() if p.is_dir()][:50])


In [ ]:
# Select the actual mounted artifact dataset. This cell never downloads weights or wheels.
NATIVE_NAME = "vllm-0.30.0-cp312-cp312-linux_x86_64.whl"
EXPECTED_NATIVE = "9ce42cde3a8564d9670679dd1be11b0a8640fb88d22aef8f94299d5a6747a6e8"
def sha256(path):
    h=hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda:stream.read(4*1024*1024),b""):h.update(block)
    return h.hexdigest()
NATIVE_CANDIDATES=[p for p in INPUT.rglob(NATIVE_NAME) if p.is_file() and not p.is_symlink()]
print("native candidates",NATIVE_CANDIDATES)
assert len(NATIVE_CANDIDATES)>=1,"Mount the P31 native-wheel Input"
NATIVE=NATIVE_CANDIDATES[0]
assert NATIVE.stat().st_size==293855397 and sha256(NATIVE)==EXPECTED_NATIVE
print("native wheel verified",NATIVE)


In [ ]:
# The SDK wheel, manifest, split ZIPs and installer are expected as mounted Inputs.
SDK_NAME="kaggle_vllm-0.3.2-py3-none-any.whl"
SDK_CANDIDATES=[p for p in INPUT.rglob(SDK_NAME) if p.is_file() and not p.is_symlink()]
MANIFEST_CANDIDATES=[p for p in INPUT.rglob("RELEASE_MANIFEST.json") if p.is_file() and not p.is_symlink()]
print("sdk",SDK_CANDIDATES,"manifests",MANIFEST_CANDIDATES)
assert len(SDK_CANDIDATES)==1 and len(MANIFEST_CANDIDATES)==1,"Mount the local candidate SDK and release manifest"
SDK=SDK_CANDIDATES[0]; MANIFEST=MANIFEST_CANDIDATES[0]
RELEASE=json.loads(MANIFEST.read_text())
assert RELEASE["sdk_version"]=="0.3.2" and RELEASE["native_wheel"]["sha256"]==EXPECTED_NATIVE
INPUT_ROOT=NATIVE.parent
for record in [*RELEASE["archives"].values(),RELEASE["sdk_dependency_archive"]]:
    assert (INPUT_ROOT/record["filename"]).is_file(),f"Missing split ZIP next to native wheel: {record['filename']}"
print("local artifacts present; this is not an installation result")


In [ ]:
# Embed the reviewed offline installer so no separate helper script can be missing.
INSTALLER=Path("/kaggle/working/install_global_v032_offline.py")
INSTALLER.write_text('#!/usr/bin/python3.12\n"""Stage the exact v0.3.2 Kaggle wheels and optionally install globally.\n\nThis script makes no network requests. Without --apply it only verifies and\nstages files. --apply also requires --replace-global-cuda-stack because pip\nwill replace Kaggle\'s preinstalled Torch/CUDA packages.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport zipfile\nfrom pathlib import Path\n\nARCHIVES = {\n    "runtime-core": "kaggle-vllm-v030-py312-runtime-core-v2.zip",\n    "runtime-large": "kaggle-vllm-v030-py312-runtime-large-v2.zip",\n    "torch-cu129": "kaggle-vllm-v030-py312-torch-cu129-v2.zip",\n}\nLOCK_MEMBER = "kaggle_vllm/profiles/kaggle-l4x4-cu129/L4_INFERENCE_LOCK.json"\n\n\ndef digest(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as source:\n        for block in iter(lambda: source.read(4 * 1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef check_file(path: Path, record: dict) -> None:\n    if path.is_symlink() or not path.is_file():\n        raise ValueError(f"required regular file is missing or symlinked: {path}")\n    if path.name != record["filename"]:\n        raise ValueError(f"filename mismatch: {path.name} != {record[\'filename\']}")\n    if path.stat().st_size != record["size"] or digest(path) != record["sha256"]:\n        raise ValueError(f"size or SHA256 mismatch: {path}")\n\n\ndef safe_stage(root: Path, release_sha: str) -> Path:\n    root = root.expanduser().absolute()\n    if root.resolve(strict=False) != root or root.is_symlink():\n        raise ValueError(f"stage path traverses a symlink: {root}")\n    if root.exists() and not root.is_dir():\n        raise ValueError(f"stage root is not a directory: {root}")\n    marker = root / "release-manifest.sha256"\n    if root.exists() and any(root.iterdir()):\n        if marker.is_symlink() or not marker.is_file() or marker.read_text().strip() != release_sha:\n            raise ValueError(f"stage root is nonempty and has another or unknown owner: {root}")\n    else:\n        root.mkdir(parents=True, exist_ok=True)\n        marker.write_text(release_sha + "\\n", encoding="ascii")\n    wheelhouse = root / "wheelhouse"\n    if wheelhouse.is_symlink():\n        raise ValueError("wheelhouse cannot be a symlink")\n    wheelhouse.mkdir(exist_ok=True)\n    return wheelhouse\n\n\ndef stage_member(archive: zipfile.ZipFile, name: str, record: dict, output: Path) -> None:\n    if Path(name).name != name or "/" in name or "\\\\" in name:\n        raise ValueError(f"unsafe ZIP member: {name}")\n    info = archive.getinfo(name)\n    if info.is_dir() or info.file_size != record["size"] or ((info.external_attr >> 16) & 0o170000) == 0o120000:\n        raise ValueError(f"unsafe or wrong-sized ZIP member: {name}")\n    if output.exists():\n        check_file(output, record)\n        return\n    if output.is_symlink():\n        raise ValueError(f"wheel target is a symlink: {output}")\n    with tempfile.NamedTemporaryFile(prefix=".wheel-", dir=output.parent, delete=False) as target:\n        temporary = Path(target.name)\n        h = hashlib.sha256()\n        count = 0\n        try:\n            with archive.open(info) as source:\n                for block in iter(lambda: source.read(4 * 1024 * 1024), b""):\n                    count += len(block)\n                    if count > record["size"]:\n                        raise ValueError(f"ZIP member exceeds pinned size: {name}")\n                    target.write(block)\n                    h.update(block)\n            target.flush()\n            os.fsync(target.fileno())\n            if count != record["size"] or h.hexdigest() != record["sha256"]:\n                raise ValueError(f"ZIP wheel hash mismatch: {name}")\n            os.replace(temporary, output)\n        finally:\n            temporary.unlink(missing_ok=True)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input-root", required=True, type=Path, help="mounted Kaggle Input containing the pinned v2 ZIPs and native wheel")\n    parser.add_argument("--sdk-wheel", required=True, type=Path)\n    parser.add_argument("--release-manifest", required=True, type=Path)\n    parser.add_argument("--stage-root", type=Path, default=Path("/tmp/kaggle-vllm/v0.3.2/global-install"))\n    parser.add_argument("--apply", action="store_true", help="run pip after all identities pass")\n    parser.add_argument("--replace-global-cuda-stack", action="store_true", help="explicitly allow replacement of Kaggle\'s preinstalled Torch/CUDA packages")\n    args = parser.parse_args()\n    if sys.implementation.name != "cpython" or sys.version_info[:2] != (3, 12) or Path(sys.executable).resolve() != Path("/usr/bin/python3.12"):\n        parser.error("run exactly /usr/bin/python3.12 (CPython 3.12)")\n    if sys.prefix != sys.base_prefix:\n        parser.error("global installation requires the ordinary Python 3.12 environment")\n    if args.apply and not args.replace_global_cuda_stack:\n        parser.error("--apply requires --replace-global-cuda-stack")\n    input_root = args.input_root.expanduser().resolve(strict=True)\n    if not input_root.is_dir():\n        parser.error("--input-root must be a directory")\n    manifest_path = args.release_manifest.expanduser().resolve(strict=True)\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n    if manifest.get("status") != "PENDING_KAGGLE_ACCEPTANCE" or manifest.get("sdk_version") != "0.3.2":\n        raise ValueError("wrong or incomplete v0.3.2 release manifest")\n    check_file(args.sdk_wheel, manifest["sdk_wheel"])\n    native = input_root / manifest["native_wheel"]["filename"]\n    check_file(native, manifest["native_wheel"])\n    with zipfile.ZipFile(args.sdk_wheel) as sdk:\n        lock_bytes = sdk.read(LOCK_MEMBER)\n    if hashlib.sha256(lock_bytes).hexdigest() != manifest["dependency_lock_sha256"]:\n        raise ValueError("SDK wheel embeds a different dependency lock")\n    lock = json.loads(lock_bytes)\n    if lock.get("wheel_count") != 168 or len(lock.get("wheels", [])) != 168 or lock.get("native") != manifest["native_wheel"]:\n        raise ValueError("SDK wheel has an unexpected P31 inference lock")\n    release_sha = digest(manifest_path)\n    wheelhouse = safe_stage(args.stage_root, release_sha)\n    needed = sum(r["size"] for r in lock["wheels"]) + manifest["sdk_dependency"]["size"]\n    if shutil.disk_usage(wheelhouse).free < needed + 10 * 1024**3:\n        raise ValueError("insufficient free disk for verified wheel staging and global pip installation")\n    groups: dict[str, list[dict]] = {name: [] for name in ARCHIVES}\n    for record in lock["wheels"]:\n        if record["group"] not in groups:\n            raise ValueError(f"unexpected wheel group: {record[\'group\']}")\n        groups[record["group"]].append(record)\n    for group, filename in ARCHIVES.items():\n        archive_record = manifest["archives"][filename]\n        archive = input_root / filename\n        check_file(archive, archive_record)\n        with zipfile.ZipFile(archive) as source:\n            for record in groups[group]:\n                stage_member(source, record["filename"], record, wheelhouse / record["filename"])\n    build_archive_name = manifest["sdk_dependency_archive"]["filename"]\n    build_archive = input_root / build_archive_name\n    check_file(build_archive, manifest["sdk_dependency_archive"])\n    packaging_record = manifest["sdk_dependency"]\n    with zipfile.ZipFile(build_archive) as source:\n        stage_member(source, packaging_record["filename"], packaging_record, wheelhouse / packaging_record["filename"])\n    requirements = [*lock["wheels"], packaging_record, lock["native"], manifest["sdk_wheel"]]\n    names = [r["filename"] for r in requirements]\n    if len(set(names)) != len(names):\n        raise ValueError("release lock contains duplicate wheel filenames")\n    lines = []\n    for record in requirements:\n        if record is lock["native"]:\n            distribution, version = "vllm", "0.30.0"\n        else:\n            distribution, version = record["distribution"], record["version"]\n        lines.append(f"{distribution}=={version} --hash=sha256:{record[\'sha256\']}")\n    target = wheelhouse.parent / "requirements-locked.txt"\n    if target.is_symlink():\n        raise ValueError("requirements target cannot be a symlink")\n    target.write_text("\\n".join(lines) + "\\n", encoding="ascii")\n    print(json.dumps({"status": "VERIFIED_STAGED", "wheel_count": len(requirements), "stage": str(wheelhouse), "apply": args.apply}, indent=2))\n    if not args.apply:\n        return 0\n    command = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "--no-cache-dir", "--no-compile", "--force-reinstall", "--only-binary=:all:", "--require-hashes", "--find-links", str(wheelhouse), "--find-links", str(native.parent), "--find-links", str(args.sdk_wheel.parent), "-r", str(target)]\n    environment = dict(os.environ, PIP_NO_INDEX="1", PIP_DISABLE_PIP_VERSION_CHECK="1")\n    subprocess.run(command, check=True, env=environment)\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',encoding="utf-8")
assert sha256(INSTALLER)=='130f470c15dab3ccbf16e58a85bdf97acaba1d96f03fd6b9eaafec192fc241ef'
print("installer",INSTALLER)


In [ ]:
# Explicitly approve replacement of Kaggle's preinstalled Torch/CUDA 12.8 stack.
# First run staging only; inspect output before setting APPLY_INSTALL=True.
APPLY_INSTALL=False
COMMAND=[PYTHON,str(INSTALLER),"--input-root",str(INPUT_ROOT),"--sdk-wheel",str(SDK),"--release-manifest",str(MANIFEST)]
if APPLY_INSTALL: COMMAND += ["--apply","--replace-global-cuda-stack"]
result=subprocess.run(COMMAND,text=True,capture_output=True,timeout=3600)
print(result.stdout[-6000:]); print(result.stderr[-3000:])
assert result.returncode==0,result.returncode
if not APPLY_INSTALL: raise RuntimeError("Staging verified. Review it, then explicitly set APPLY_INSTALL=True and rerun this cell before continuing.")


In [ ]:
# Confirm installed distributions and imports in the SAME CPython 3.12 environment.
probe=r'''import importlib,importlib.metadata as m,json,sys,traceback
names=['kaggle-vllm','vllm','torch','triton']
result={'python':sys.executable,'distributions':{n:m.version(n) for n in names}}
import torch,triton,vllm
from vllm import LLM
import vllm._C_stable_libtorch
import vllm.vllm_flash_attn._vllm_fa2_C
result.update(torch_cuda=torch.version.cuda,modules={n:importlib.import_module(n).__file__ for n in ['kaggle_vllm','vllm','torch','triton']},native_imports='PASS')
print(json.dumps(result,indent=2))'''
check=subprocess.run([PYTHON,"-c",probe],capture_output=True,text=True,timeout=180)
print(check.stdout[-6000:]);print(check.stderr[-3000:]);assert check.returncode==0


In [ ]:
# Strict global SDK doctor; a native import does not establish engine or model success.
doctor=subprocess.run([PYTHON,"-c","from kaggle_vllm.cli_v3 import main; raise SystemExit(main())","doctor","--profile","kaggle-l4x4-cu129","--runtime-mode","global","--strict","--import-native","--json"],capture_output=True,text=True,timeout=180)
print(doctor.stdout[-10000:]);print(doctor.stderr[-3000:]);assert doctor.returncode==0


In [ ]:
# Verify NCCL all-reduce across exactly four L4 processes.
nccl_code=r'''import os,torch,torch.distributed as dist
dist.init_process_group('nccl')
r=dist.get_rank();torch.cuda.set_device(r)
x=torch.tensor([r+1],device=f'cuda:{r}',dtype=torch.float32)
dist.all_reduce(x)
assert x.item()==10.0,(r,x.item())
print(f'rank={r} sum={x.item()}',flush=True)
dist.destroy_process_group()'''
NCCL_SCRIPT=Path('/kaggle/working/nccl_v032_manual.py');NCCL_SCRIPT.write_text(nccl_code)
nc=subprocess.run([PYTHON,'-m','torch.distributed.run','--standalone','--nproc_per_node=4',str(NCCL_SCRIPT)],capture_output=True,text=True,timeout=240)
print(nc.stdout[-4000:]);print(nc.stderr[-4000:]);assert nc.returncode==0


In [ ]:
# Discover mounted Gemma 4 QAT checkpoint. Select the real local path if several match.
MODEL_CANDIDATES=[p.parent for p in INPUT.rglob('config.json') if p.is_file() and not p.is_symlink() and 'gemma-4-31b-it-qat-w4a16-ct' in str(p.parent).lower()]
print('model candidates',MODEL_CANDIDATES)
assert len(MODEL_CANDIDATES)==1,'Set MODEL manually to the actual mounted model folder with config.json'
MODEL=MODEL_CANDIDATES[0]
assert MODEL.is_dir() and (MODEL/'config.json').is_file()


In [ ]:
# Real TP4 generation runs in a child process so timeout and cleanup are explicit.
infer_code=r'''import json,os,sys,traceback
from kaggle_vllm import KaggleLLM,KaggleSamplingParams
model=sys.argv[1]
llm=KaggleLLM(model,profile='kaggle-l4x4-cu129',runtime_mode='global',strict=True,tensor_parallel_size=4,dtype='bfloat16',max_model_len=2048,gpu_memory_utilization=0.86)
result=llm.generate(['Explain tensor parallelism in one sentence.'],KaggleSamplingParams(temperature=0,max_tokens=64))
print(json.dumps({'output_count':len(result),'text':[o.outputs[0].text for o in result],'selected_attention_backend':llm.effective_attention_backend()},ensure_ascii=False),flush=True)'''
INFER_SCRIPT=Path('/kaggle/working/gemma4_tp4_v032_manual.py');INFER_SCRIPT.write_text(infer_code)
process=subprocess.Popen([PYTHON,str(INFER_SCRIPT),str(MODEL)],stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True,start_new_session=True)
try: out,err=process.communicate(timeout=1200)
except subprocess.TimeoutExpired:
    os.killpg(process.pid,signal.SIGTERM)
    try: out,err=process.communicate(timeout=20)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid,signal.SIGKILL);out,err=process.communicate()
    raise
print(out[-6000:]);print(err[-6000:]);assert process.returncode==0


In [ ]:
# Export small evidence, including actual backend trace tail; never claim a backend from extension import alone.
EVIDENCE=Path('/kaggle/working/kaggle_vllm_v032_manual_evidence');EVIDENCE.mkdir(exist_ok=True)
(EVIDENCE/'gpu.txt').write_text(sm.stdout)
(EVIDENCE/'imports.txt').write_text(check.stdout[-10000:]+'\n'+check.stderr[-4000:])
(EVIDENCE/'doctor.json').write_text(doctor.stdout)
(EVIDENCE/'nccl.txt').write_text(nc.stdout[-10000:]+'\n'+nc.stderr[-4000:])
(EVIDENCE/'inference.txt').write_text(out[-10000:]+'\n'+err[-10000:])
(EVIDENCE/'manifest.sha256').write_text(sha256(MANIFEST)+'\n')
ZIP=Path('/kaggle/working/kaggle_vllm_v032_manual_evidence.zip')
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in EVIDENCE.iterdir():archive.write(path,arcname=path.name)
print('Download',ZIP)
